In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')
        

/kaggle/input/datasets/kaushikar/drone-usat/DIAT-uSAT_dataset/Bird+mini-helicopter_1/Bird+2_Blade_rotor_1/figure370.jpg
/kaggle/input/datasets/kaushikar/drone-usat/DIAT-uSAT_dataset/Bird+mini-helicopter_1/Bird+2_Blade_rotor_1/figure190.jpg
/kaggle/input/datasets/kaushikar/drone-usat/DIAT-uSAT_dataset/Bird+mini-helicopter_1/Bird+2_Blade_rotor_1/figure175.jpg
/kaggle/input/datasets/kaushikar/drone-usat/DIAT-uSAT_dataset/Bird+mini-helicopter_1/Bird+2_Blade_rotor_1/figure287.jpg
/kaggle/input/datasets/kaushikar/drone-usat/DIAT-uSAT_dataset/Bird+mini-helicopter_1/Bird+2_Blade_rotor_1/figure135.jpg
/kaggle/input/datasets/kaushikar/drone-usat/DIAT-uSAT_dataset/Bird+mini-helicopter_1/Bird+2_Blade_rotor_1/figure121.jpg
/kaggle/input/datasets/kaushikar/drone-usat/DIAT-uSAT_dataset/Bird+mini-helicopter_1/Bird+2_Blade_rotor_1/figure149.jpg
/kaggle/input/datasets/kaushikar/drone-usat/DIAT-uSAT_dataset/Bird+mini-helicopter_1/Bird+2_Blade_rotor_1/figure265.jpg
/kaggle/input/datasets/kaushikar/drone-u

In [2]:
import os
import glob
import time
import random
import subprocess
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             precision_recall_fscore_support, classification_report,
                             confusion_matrix, cohen_kappa_score, matthews_corrcoef,
                             roc_auc_score)

try:
    from thop import profile as thop_profile
except Exception:
    subprocess.run(['pip', 'install', '-q', 'thop'])
    from thop import profile as thop_profile

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = '/kaggle/input/datasets/kaushikar/drone-usat/DIAT-uSAT_dataset'
WORK = '/kaggle/working'
CKPT = os.path.join(WORK, 'axis_decoupled_net.pt')
IMG_H, IMG_W = 176, 224
BATCH = 64
EPOCHS = 60
LR = 1e-3

CLASS_MAP = {
    '3_long_blade_rotor': '3_long_blade_rotor',
    '3_short_blade_rotor_1': '3_short_blade_rotor',
    '3_short_blade_rotor_2': '3_short_blade_rotor',
    'Bird': 'Bird',
    'Bird+mini-helicopter_1': 'Bird+mini-helicopter',
    'Bird+mini-helicopter_2': 'Bird+mini-helicopter',
    'RC plane_1': 'RC_plane',
    'RC plane_2': 'RC_plane',
    'drone_1': 'drone',
    'drone_2': 'drone',
}
CLASSES = sorted(set(CLASS_MAP.values()))
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 83.8 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

In [3]:
def list_images(folder):
    exts = ('*.png', '*.jpg', '*.jpeg', '*.bmp', '*.PNG', '*.JPG', '*.JPEG')
    files = []
    for e in exts:
        files += glob.glob(os.path.join(folder, '**', e), recursive=True)
    return files


def autocrop_resize(pil_img):
    a = np.asarray(pil_img.convert('L'))
    mask = a < 240
    if mask.any():
        rows = np.where(mask.any(axis=1))[0]
        cols = np.where(mask.any(axis=0))[0]
        a = a[rows[0]:rows[-1] + 1, cols[0]:cols[-1] + 1]
    out = Image.fromarray(a).resize((IMG_W, IMG_H), Image.BILINEAR)
    return np.asarray(out, dtype=np.uint8)


paths, labels = [], []
for top in sorted(os.listdir(ROOT)):
    if top in CLASS_MAP:
        c = CLASS_TO_IDX[CLASS_MAP[top]]
        for f in list_images(os.path.join(ROOT, top)):
            paths.append(f)
            labels.append(c)
labels = np.array(labels, dtype=np.int64)

print('Total images:', len(paths))
for c in CLASSES:
    print(f'  {c}: {(labels == CLASS_TO_IDX[c]).sum()}')

data = np.zeros((len(paths), IMG_H, IMG_W), dtype=np.uint8)
for i, p in enumerate(paths):
    data[i] = autocrop_resize(Image.open(p))

idx = np.arange(len(paths))
tr_idx, tmp_idx = train_test_split(idx, test_size=0.2, stratify=labels, random_state=SEED)
va_idx, te_idx = train_test_split(tmp_idx, test_size=0.5, stratify=labels[tmp_idx], random_state=SEED)
print('Train/Val/Test:', len(tr_idx), len(va_idx), len(te_idx))

np.save(os.path.join(WORK, 'test_images.npy'), data[te_idx])
np.save(os.path.join(WORK, 'test_labels.npy'), labels[te_idx])
np.save(os.path.join(WORK, 'classes.npy'), np.array(CLASSES))
print('Held-out test set saved:', data[te_idx].shape)

Total images: 4849
  3_long_blade_rotor: 799
  3_short_blade_rotor: 800
  Bird: 800
  Bird+mini-helicopter: 815
  RC_plane: 800
  drone: 835
Train/Val/Test: 3879 485 485
Held-out test set saved: (485, 176, 224)


In [4]:
class ConvBNAct(nn.Module):
    def __init__(self, ci, co, k=3, s=1):
        super().__init__()
        self.c = nn.Conv2d(ci, co, k, s, k // 2, bias=False)
        self.bn = nn.BatchNorm2d(co)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.c(x)))


class DSConv(nn.Module):
    def __init__(self, ci, co, s=1):
        super().__init__()
        self.dw = nn.Conv2d(ci, ci, 3, s, 1, groups=ci, bias=False)
        self.pw = nn.Conv2d(ci, co, 1, 1, 0, bias=False)
        self.bn = nn.BatchNorm2d(co)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.pw(self.dw(x))))


class Conv1dBNAct(nn.Module):
    def __init__(self, ci, co, k=5, s=2):
        super().__init__()
        self.c = nn.Conv1d(ci, co, k, s, k // 2, bias=False)
        self.bn = nn.BatchNorm1d(co)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.c(x)))


class AxisDecoupledNet(nn.Module):
    def __init__(self, n_classes=6, width=32):
        super().__init__()
        self.stem = nn.Sequential(
            ConvBNAct(1, 16, 3, 2),
            ConvBNAct(16, width, 3, 2),
        )
        self.joint = nn.Sequential(
            DSConv(width, 48, s=2),
            DSConv(48, 64, s=2),
        )
        self.spectral = nn.Sequential(
            Conv1dBNAct(width, 48, 5, 2),
            Conv1dBNAct(48, 64, 5, 2),
        )
        self.temporal = nn.Sequential(
            Conv1dBNAct(width, 48, 5, 2),
            Conv1dBNAct(48, 64, 5, 2),
        )
        self.gap2d = nn.AdaptiveAvgPool2d(1)
        self.gap1d = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Linear(192, 96),
            nn.BatchNorm1d(96),
            nn.SiLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(96, n_classes),
        )

    def forward(self, x):
        f = self.stem(x)
        j = self.gap2d(self.joint(f)).flatten(1)
        s = self.gap1d(self.spectral(f.mean(dim=3))).flatten(1)
        t = self.gap1d(self.temporal(f.mean(dim=2))).flatten(1)
        return self.head(torch.cat([j, s, t], dim=1))

In [5]:
class MDDataset(Dataset):
    def __init__(self, arr, lab, indices, train=False):
        self.arr = arr
        self.lab = lab
        self.idx = indices
        self.train = train

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        j = self.idx[i]
        img = self.arr[j].astype(np.float32) / 255.0
        if self.train:
            if random.random() < 0.5:
                img = img[:, ::-1].copy()
            if random.random() < 0.5:
                img = np.clip(img * random.uniform(0.9, 1.1), 0.0, 1.0)
            if random.random() < 0.3:
                f0 = random.randint(0, IMG_H - 20)
                img[f0:f0 + random.randint(5, 20), :] = 0.0
            if random.random() < 0.3:
                t0 = random.randint(0, IMG_W - 25)
                img[:, t0:t0 + random.randint(5, 25)] = 0.0
        return torch.from_numpy(img).unsqueeze(0), int(self.lab[j])


train_loader = DataLoader(MDDataset(data, labels, tr_idx, True), batch_size=BATCH,
                          shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(MDDataset(data, labels, va_idx, False), batch_size=BATCH,
                        shuffle=False, num_workers=2, pin_memory=True)

model = AxisDecoupledNet(n_classes=len(CLASSES)).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)
crit = nn.CrossEntropyLoss(label_smoothing=0.05)
scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')

best_acc = 0.0
for ep in range(EPOCHS):
    model.train()
    run = 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        opt.zero_grad()
        with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
            out = model(x)
            loss = crit(out, y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        run += loss.item() * x.size(0)
    sched.step()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            p = model(x).argmax(1)
            correct += (p == y).sum().item()
            total += y.size(0)
    va = correct / total
    if va > best_acc:
        best_acc = va
        torch.save(model.state_dict(), CKPT)
    print(f'Epoch {ep + 1:02d}/{EPOCHS}  train_loss {run / len(tr_idx):.4f}  val_acc {va:.4f}')

print('Best val_acc:', round(best_acc, 4), '| checkpoint:', CKPT)

Epoch 01/60  train_loss 0.9699  val_acc 0.5485
Epoch 02/60  train_loss 0.6441  val_acc 0.8887
Epoch 03/60  train_loss 0.5471  val_acc 0.8577
Epoch 04/60  train_loss 0.5012  val_acc 0.8845
Epoch 05/60  train_loss 0.4510  val_acc 0.8722
Epoch 06/60  train_loss 0.4383  val_acc 0.7897
Epoch 07/60  train_loss 0.4080  val_acc 0.9361
Epoch 08/60  train_loss 0.4016  val_acc 0.9196
Epoch 09/60  train_loss 0.3764  val_acc 0.9010
Epoch 10/60  train_loss 0.3672  val_acc 0.9546
Epoch 11/60  train_loss 0.3546  val_acc 0.9443
Epoch 12/60  train_loss 0.3512  val_acc 0.9423
Epoch 13/60  train_loss 0.3431  val_acc 0.9691
Epoch 14/60  train_loss 0.3300  val_acc 0.9794
Epoch 15/60  train_loss 0.3346  val_acc 0.9670
Epoch 16/60  train_loss 0.3226  val_acc 0.9732
Epoch 17/60  train_loss 0.3256  val_acc 0.9649
Epoch 18/60  train_loss 0.3137  val_acc 0.9320
Epoch 19/60  train_loss 0.3068  val_acc 0.9773
Epoch 20/60  train_loss 0.3013  val_acc 0.9732
Epoch 21/60  train_loss 0.3004  val_acc 0.9670
Epoch 22/60  

In [6]:
test_images = np.load(os.path.join(WORK, 'test_images.npy'))
test_labels = np.load(os.path.join(WORK, 'test_labels.npy'))
class_names = list(np.load(os.path.join(WORK, 'classes.npy')))

infer_model = AxisDecoupledNet(n_classes=len(class_names)).to(DEVICE)
infer_model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
infer_model.eval()

y_true = test_labels
y_prob = []
with torch.no_grad():
    for i in range(0, len(test_images), BATCH):
        b = test_images[i:i + BATCH].astype(np.float32) / 255.0
        xb = torch.from_numpy(b).unsqueeze(1).to(DEVICE)
        out = infer_model(xb)
        y_prob.append(torch.softmax(out, 1).cpu().numpy())
y_prob = np.concatenate(y_prob)
y_pred = y_prob.argmax(1)

acc = accuracy_score(y_true, y_pred)
bacc = balanced_accuracy_score(y_true, y_pred)
pr_m, rc_m, f1_m, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
pr_w, rc_w, f1_w, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
kappa = cohen_kappa_score(y_true, y_pred)
mcc = matthews_corrcoef(y_true, y_pred)
try:
    auc = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')
except Exception:
    auc = float('nan')

n_params = sum(p.numel() for p in infer_model.parameters())
size_mb = os.path.getsize(CKPT) / 1e6

dummy = torch.randn(1, 1, IMG_H, IMG_W).to(DEVICE)
macs, _ = thop_profile(infer_model, inputs=(dummy,), verbose=False)
flops_g = 2 * macs / 1e9
macs_m = macs / 1e6

with torch.no_grad():
    for _ in range(20):
        _ = infer_model(dummy)
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(200):
        _ = infer_model(dummy)
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    lat_ms = (time.time() - t0) / 200 * 1000

    batch_dummy = torch.randn(BATCH, 1, IMG_H, IMG_W).to(DEVICE)
    for _ in range(10):
        _ = infer_model(batch_dummy)
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(50):
        _ = infer_model(batch_dummy)
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    thr = BATCH * 50 / (time.time() - t0)

print('============ INFERENCE FROM CHECKPOINT ============')
print(f'Checkpoint             : {CKPT}')
print(f'Device                 : {torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"}')
print(f'Test samples           : {len(y_true)}')
print(f'Test Accuracy          : {acc * 100:.2f}%')
print(f'Balanced Accuracy      : {bacc * 100:.2f}%')
print(f'Precision (macro/wtd)  : {pr_m:.4f} / {pr_w:.4f}')
print(f'Recall    (macro/wtd)  : {rc_m:.4f} / {rc_w:.4f}')
print(f'F1-score  (macro/wtd)  : {f1_m:.4f} / {f1_w:.4f}')
print(f"Cohen's Kappa          : {kappa:.4f}")
print(f'Matthews CorrCoef      : {mcc:.4f}')
print(f'ROC-AUC (macro OvR)    : {auc:.4f}')
print(f'Total Parameters       : {n_params:,}')
print(f'Checkpoint Size        : {size_mb:.3f} MB')
print(f'MACs                   : {macs_m:.3f} M')
print(f'FLOPs                  : {flops_g:.4f} G')
print(f'Latency (batch=1)      : {lat_ms:.3f} ms')
print(f'Throughput (batch={BATCH})  : {thr:.1f} img/s')
print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=class_names, digits=4, zero_division=0))
print('Confusion Matrix:')
print(confusion_matrix(y_true, y_pred))

============ INFERENCE FROM CHECKPOINT ============
Checkpoint             : /kaggle/working/axis_decoupled_net.pt
Device                 : Tesla T4
Test samples           : 485
Test Accuracy          : 99.38%
Balanced Accuracy      : 99.38%
Precision (macro/wtd)  : 0.9937 / 0.9938
Recall    (macro/wtd)  : 0.9938 / 0.9938
F1-score  (macro/wtd)  : 0.9937 / 0.9938
Cohen's Kappa          : 0.9926
Matthews CorrCoef      : 0.9926
ROC-AUC (macro OvR)    : 0.9999
Total Parameters       : 76,230
Checkpoint Size        : 0.330 MB
MACs                   : 16.355 M
FLOPs                  : 0.0327 G
Latency (batch=1)      : 1.251 ms
Throughput (batch=64)  : 22065.7 img/s

Classification Report:
                      precision    recall  f1-score   support

  3_long_blade_rotor     0.9877    1.0000    0.9938        80
 3_short_blade_rotor     0.9873    0.9750    0.9811        80
                Bird     1.0000    1.0000    1.0000        80
Bird+mini-helicopter     1.0000    1.0000    1.0000        